In [ ]:
# Setup — all imports live here so the notebook executes top-to-bottom
# without reimporting mid-notebook. When running headlessly (CI, HPC),
# uncomment the ``matplotlib.use('Agg')`` line *before* the pyplot import
# so figures never need an interactive display.
import dataclasses
import logging
import os
import sys
from pathlib import Path

# import matplotlib
# matplotlib.use('Agg')  # enable for headless runs
import matplotlib.pyplot as plt
import mne
import numpy as np
import seaborn as sns
from mne.viz import plot_topomap

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:  # noqa: B007
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

from sklearn.decomposition import PCA, FastICA  # noqa: E402

from scripts.analysis_common import (  # noqa: E402
    FREQUENCY_BANDS,
    WAVELET_BAND_FREQ_RESOLUTION_HZ,
    analyzers_to_datasets,
    load_analyzers,
    wavelet_transform,
)
from src.analysis.wavelet_ica import zscore_by_time  # noqa: E402
from src.definitions.constants import ProjectPaths  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    ConditionVariants,
    ExclusionCategories,
    MusicTypeVariants,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
%matplotlib inline

# Freq–Channel Features ICA on Wavelet Power

## Scope

This notebook performs ICA decomposition on wavelet power data where
**frequencies and channels** form the observation axis, and **subjects
and time** are combined into the feature axis:

```
Input:   (n_subjects, n_channels, n_freqs, n_times)  — 4-D wavelet power
Reshape: (n_freqs × n_channels,  n_subjects × n_times)
         ──── observations ─────  ──── features ──────
```

Before reshaping the tensor is **z-scored along the time axis** so that
every `(subject, channel, frequency)` slice has zero mean and unit
variance.  This removes overall amplitude differences and ensures that
PCA/ICA operates on standardised activations.

## What the decomposition finds

Each observation is a specific **frequency–channel combination**, described
by its full subject × time power surface.  PCA followed by ICA discovers
a small set of **subject-temporal component patterns** (each of length
S × T) — recurring subject × time patterns shared across frequencies and
channels.  The ICA score for each component can be reshaped back to
`(n_freqs, n_channels)` and decomposed into:

| Quantity | Shape | Interpretation |
|----------|-------|----------------|
| **Subject-temporal component** | `(S, T)` | A subject × time pattern shared across observations |
| **Subject loadings** | `(S,)` | Time-averaged participation — which participants contribute most |
| **Temporal profile** | `(T,)` | Subject-averaged time course of the component |
| **Frequency loadings** | `(F,)` | Mean |score| over channels — which bands dominate |
| **Channel loadings** | `(C,)` | Mean |score| over frequencies — spatial topography |

## Why this reshape?

By combining frequencies and channels into observations while keeping
subjects and time as features, this decomposition is optimised for
finding **subject-temporal modes** — recurring temporal activation
patterns that characterise how individual participants respond to the
stimulus.  This is complementary to the subject-freq-features approach
(Approach 8), which finds spatial-temporal modes, and the combined-features
approach (Approach 6), which finds temporal patterns.

The key advantage is that the observation axis (F × C) groups the data
by *which frequency* and *which electrode*, allowing ICA to discover
subject × time fingerprints that generalise across both spectral bands
and scalp locations.  If a component has high scores for many frequencies
and channels, it represents a robust, stimulus-driven subject-temporal mode.

## Analyses

1. Z-scoring and reshape
2. PCA dimensionality reduction + ICA decomposition
3. **(a)** Intersubject correlation matrix of ICA components
4. **(b)** Component temporal profiles — mean ± std across subjects
5. **(c)** Frequency × Time mean-loading heatmaps of component activations
6. **(d)** Mean and variance of component channel loadings as topomaps
7. **(e)** Per-subject loading bar plot for each component

## Configuration

In [ ]:
# ── Experiment configuration ─────────────────────────────────────────────────
CONDITION = ConditionVariants.PLACEBO
MUSIC_TYPES = [MusicTypeVariants.CLASSICAL]  # single type for fast exploration
EXCLUSION_CATEGORIES = [ExclusionCategories.BAD_MUSIC, ExclusionCategories.ARTIFACTS]
PROCESS_AND_SAVE_DATA = False  # set True to re-process raw files

# ── Wavelet settings ─────────────────────────────────────────────────────────
REPRESENTATION = "power"
WAVELET_FREQ_MIN = min(lo for lo, _ in FREQUENCY_BANDS.values())
WAVELET_FREQ_MAX = max(hi for _, hi in FREQUENCY_BANDS.values())
WAVELET_N_FREQS = max(
    2,
    int(round((WAVELET_FREQ_MAX - WAVELET_FREQ_MIN) / WAVELET_BAND_FREQ_RESOLUTION_HZ))
    + 1,
)
FREQS = np.linspace(WAVELET_FREQ_MIN, WAVELET_FREQ_MAX, WAVELET_N_FREQS)

KEEP_FREQUENCY_DIM = True
RESHAPE_FREQUENCY_DIM = True  # -> (n_subjects, n_channels, n_freqs, n_times)

# ── Reuse / compute ──────────────────────────────────────────────────────────
REUSE_WAVELETS = True  # load from cache; set False to compute + save

# ── Subject subset ────────────────────────────────────────────────────────────
N_SUBJECTS_SUBSET: int | None = 5

# ── Channel and time subset ───────────────────────────────────────────────────
N_CHANNELS_SUBSET: int | None = 32  # first N channels (from 195)
N_TIMES_SUBSET: int | None = 10000  # first N time samples

# ── Decomposition settings ────────────────────────────────────────────────────
N_COMPONENTS_PCA = 20  # number of PCA components to retain
N_COMPONENTS_ICA = 10  # number of ICA components to extract
ICA_RANDOM_STATE = 42  # reproducibility

# ── Storage directory ─────────────────────────────────────────────────────────
WAVELET_DIR: Path = (
    ProjectPaths.NOTEBOOKS_DIR / "04-wavelet-ica-analysis" / "wavelet_cache"
)

# ── Plot saving ──────────────────────────────────────────────────────────────
SAVE_PLOTS = True
PLOTS_DIR = (
    ProjectPaths.NOTEBOOKS_DIR
    / "04-wavelet-ica-analysis"
    / "plots"
    / "freq_channel_features"
    / "pca_ica"
)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Wavelet base directory : {WAVELET_DIR}")
print(
    f"Frequencies            : {FREQS[0]:.1f}\u2013{FREQS[-1]:.1f} Hz ({len(FREQS)} steps)"
)
print(f"PCA components         : {N_COMPONENTS_PCA}")
print(f"ICA components         : {N_COMPONENTS_ICA}")

## Data Loading

In [ ]:
analyzers = load_analyzers(
    MUSIC_TYPES,
    CONDITION,
    EXCLUSION_CATEGORIES,
    PROCESS_AND_SAVE_DATA,
    normalize_data=False,
)
datasets = analyzers_to_datasets(analyzers)

# Always limit to first N_SUBJECTS_SUBSET individuals
if N_SUBJECTS_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:N_SUBJECTS_SUBSET])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_SUBJECTS_SUBSET} individuals.")

# Slice to channel subset
if N_CHANNELS_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:, :N_CHANNELS_SUBSET, :])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_CHANNELS_SUBSET} channels.")

# Slice to time subset
if N_TIMES_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:, :, :N_TIMES_SUBSET])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_TIMES_SUBSET} time samples.")

print("Loaded datasets:", list(datasets.keys()))
for label, ad in datasets.items():
    print(
        f"  {label}: {ad.n_items} subjects, {ad.n_features} channels, "
        f"{ad.n_samples} samples"
    )

## Load or Compute Wavelet Transforms

Stored in `WAVELET_DIR/broadband/`.

In [ ]:
broadband_datasets = wavelet_transform(
    datasets=datasets,
    freqs=FREQS,
    representation=REPRESENTATION,
    keep_frequency_dim=KEEP_FREQUENCY_DIM,
    reshape_frequency_dim=RESHAPE_FREQUENCY_DIM,
    wavelet_dir=WAVELET_DIR / "broadband",
    reuse_wavelets=REUSE_WAVELETS,
)
for label, ad in broadband_datasets.items():
    source = ad.metadata.get("loaded_from_wavelet_file", "computed_now")
    print(f"[broadband] {label}: shape={ad.data.shape}  source={source}")

## Dataset Selection

Change `LABEL` to switch between music types.  The remaining cells use
`bb_data` (broadband wavelet power, 4-D) and derived quantities.

In [ ]:
LABEL = list(broadband_datasets.keys())[0]

bb_ad = broadband_datasets[LABEL]
bb_data = bb_ad.data  # (n_subjects, n_channels, n_freqs, n_times)
sfreq = bb_ad.sfreq

n_subjects, n_channels, n_freqs, n_times = bb_data.shape
time = np.arange(n_times) / sfreq

print(f"Dataset    : {LABEL}")
print(
    f"Shape      : {bb_data.shape}  (subjects \u00d7 channels \u00d7 freqs \u00d7 times)"
)
print(f"Duration   : {n_times / sfreq:.1f} s  @  {sfreq} Hz")
print(f"Freq range : {FREQS[0]:.1f}\u2013{FREQS[-1]:.1f} Hz ({n_freqs} steps)")

---
## Step 1 — Z-score and Reshape

**Z-scoring** normalises each `(subject, channel, frequency)` time series
to zero mean and unit variance.  This ensures that PCA/ICA are not
dominated by high-power channels, subjects, or frequency bands.

**Reshaping** combines frequencies and channels into the observation axis,
and subjects and time into the feature axis:

```
(S, C, F, T)  →  transpose to  (F, C, S, T)
              →  reshape to     (F × C,  S × T)
                                observations  features
```

Each row of the resulting 2-D matrix is the z-scored power across all
subjects and time points for a single frequency at a single channel.
PCA/ICA will discover **subject-temporal patterns** — subject × time
fingerprints — shared across frequencies and channels.

### Why this reshape?

By treating each (frequency, channel) pair as an independent observation:

- We discover **subject × time patterns** that recur across both
  spectral bands and scalp locations — individual response modes.
- The observation-to-feature ratio (F×C vs S×T) determines the
  statistical regime of the decomposition.
- Scores can be reshaped to `(F, C, K)` to reveal which frequencies
  and which channels activate each subject-temporal mode.

In [ ]:
# Z-score along time: each (subject, channel, frequency) slice → mean=0, std=1
bb_z = zscore_by_time(bb_data)  # (S, C, F, T)

# Reshape: (S, C, F, T) → transpose → (F, C, S, T) → (F*C, S*T)
bb_z_fc = bb_z.transpose(2, 1, 0, 3)  # (F, C, S, T)
n_obs = n_freqs * n_channels
n_feat = n_subjects * n_times
X_fc = bb_z_fc.reshape(n_obs, n_feat)  # (F*C, S*T)

print(f"Reshaped matrix shape : {X_fc.shape}")
print(f"  Observations (F\u00d7C)  : {X_fc.shape[0]}")
print(f"  Features     (S\u00d7T)  : {X_fc.shape[1]}")
print(f"Row means  \u2248 0 : {X_fc.mean(axis=1).mean():.6f}")
print(f"Row stds        : {X_fc.std(axis=1).mean():.4f}")

---
## Step 2 — PCA Dimensionality Reduction + ICA Decomposition

We first reduce the `S × T` feature space to `N_COMPONENTS_PCA`
principal components, keeping the directions of maximum variance.  Then
FastICA rotates the PCA subspace to maximise statistical independence,
yielding `N_COMPONENTS_ICA` independent components.

**Results:**

| Object | Shape | Description |
|--------|-------|-------------|
| `ica_scores` | `(F×C, K)` | Per-observation weight for each IC |
| `ica_components` | `(K, S×T)` | Subject-temporal pattern of each IC |
| `scores_2d` | `(F, C, K)` | ICA scores reshaped to frequency × channel |
| `components_2d` | `(K, S, T)` | ICA components reshaped to subject × time |

In [ ]:
# --- PCA ---
pca = PCA(n_components=N_COMPONENTS_PCA, random_state=ICA_RANDOM_STATE)
pca_scores = pca.fit_transform(X_fc)  # (F*C, K_pca)

explained = pca.explained_variance_ratio_
cumulative = np.cumsum(explained)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(range(1, len(explained) + 1), explained, color="steelblue")
axes[0].set_xlabel("Component")
axes[0].set_ylabel("Variance explained")
axes[0].set_title(f"PCA Scree Plot \u2014 {LABEL}")

axes[1].plot(range(1, len(cumulative) + 1), cumulative, "o-", color="coral")
axes[1].axhline(0.9, ls="--", color="gray", label="90%")
axes[1].set_xlabel("Number of components")
axes[1].set_ylabel("Cumulative variance explained")
axes[1].set_title(f"Cumulative Variance \u2014 {LABEL}")
axes[1].legend()

fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "pca_scree.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

print(
    f"Top {N_COMPONENTS_PCA} components explain "
    f"{cumulative[-1] * 100:.1f}% of total variance."
)

# --- ICA ---
ica = FastICA(
    n_components=N_COMPONENTS_ICA,
    random_state=ICA_RANDOM_STATE,
    max_iter=500,
    whiten="unit-variance",
)
ica_scores = ica.fit_transform(pca_scores)  # (F*C, K_ica)
ica_components = ica.components_ @ pca.components_  # (K_ica, S*T)

# Reshape ICA scores to (F, C, K) for downstream analysis
scores_2d = ica_scores.reshape(n_freqs, n_channels, N_COMPONENTS_ICA)  # (F, C, K)

# Reshape ICA components to (K, S, T) for subject-temporal analysis
components_2d = ica_components.reshape(
    N_COMPONENTS_ICA, n_subjects, n_times
)  # (K, S, T)

print(f"ICA scores shape       : {ica_scores.shape}")
print(f"ICA components shape   : {ica_components.shape}")
print(f"Scores 2-D shape       : {scores_2d.shape}  (F, C, K)")
print(f"Components 2-D shape   : {components_2d.shape}  (K, S, T)")

---
## Analysis (a) — Intersubject Correlation Matrix of ICA Components

For each ICA component we compute a **subject × subject** Pearson
correlation matrix.  Each subject is represented by their
**temporal component pattern** (shape `T`, i.e. the ICA component
slice for that subject).

High off-diagonal correlations indicate that the component captures a
consistent temporal activation pattern across individuals — a
hallmark of stimulus-driven (rather than noise-driven) modes.

### How this differs from subject-freq-features (Approach 8)

In the subject-freq-features notebook, each subject's loading vector
spans `F` (frequencies), because channels live in the component pattern.
Here the component pattern itself contains the subject dimension
`(S, T)`, so each subject's temporal profile IS part of the component.
The ISC matrix measures **temporal consistency** — whether subjects
show the same time course within each mode.

In [ ]:
# Per-subject temporal profile for each IC: components_2d (K, S, T)
n_show = min(6, N_COMPONENTS_ICA)
fig, axes = plt.subplots(
    1, n_show, figsize=(3.5 * n_show, 3.5), constrained_layout=True
)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    # Each subject's (T,) temporal profile for IC i
    corr_mat = np.corrcoef(components_2d[i])  # (S, S)
    im = ax.imshow(corr_mat, vmin=-1, vmax=1, cmap="RdBu_r")
    ax.set_xticks(range(n_subjects))
    ax.set_yticks(range(n_subjects))
    ax.set_xticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=7)
    ax.set_yticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=7)
    ax.set_title(f"IC {i + 1}", fontsize=10)

fig.suptitle(
    f"Intersubject Correlation of IC Temporal Profiles \u2014 {LABEL}",
    fontsize=12,
)
plt.colorbar(im, ax=axes[-1], label="Pearson r", shrink=0.8)
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "isc_component_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

---
## Analysis (b) — Component Temporal Profiles (Mean ± Std Across Subjects)

Each ICA component has a subject-temporal pattern `(S, T)`.  The
**temporal profile** is the mean across subjects → `(T,)`, and the
standard deviation captures inter-individual variability.

We also compute per-subject weighted temporal activations by combining
the component's subject-time pattern with the frequency-channel scores.
For subject *s* and component *k*:

$$a_{s,k}(t) = \text{comp}_{k,s}(t) \cdot \frac{1}{F \cdot C}\sum_{f,c} |\text{score}_{f,c,k}|$$

where the score term acts as a global weighting factor.  Since the
temporal profile is already subject-specific in this decomposition,
we can directly use the component's `(S, T)` slices.

We plot the **mean** across subjects with ± 1 std bands.

### How this differs from subject-freq-features (Approach 8)

In the subject-freq-features notebook, the component is `(C, T)` and
per-subject variability comes from projecting scores through channel
weights.  Here, the component already contains per-subject time
courses `(S, T)`, so temporal variability is directly visible in the
component itself — no projection needed.

In [ ]:
# Component temporal profiles are directly available from components_2d: (K, S, T)
mean_temporal = components_2d.mean(axis=1)  # (K, T) — mean across subjects
std_temporal = components_2d.std(axis=1)  # (K, T) — std across subjects

n_show = min(6, N_COMPONENTS_ICA)
fig, axes = plt.subplots(n_show, 1, figsize=(14, 2.5 * n_show), sharex=True)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    ax.plot(time, mean_temporal[i], lw=0.8, color="darkorange", label="mean")
    ax.fill_between(
        time,
        mean_temporal[i] - std_temporal[i],
        mean_temporal[i] + std_temporal[i],
        alpha=0.25,
        color="darkorange",
        label="\u00b1 1 std",
    )
    ax.set_ylabel(f"IC {i + 1}")
    ax.set_title(f"Component {i + 1} \u2014 Temporal Profile", fontsize=10)
    if i == 0:
        ax.legend(loc="upper right", fontsize=8)

axes[-1].set_xlabel("Time (s)")
fig.suptitle(
    f"ICA Component Temporal Profiles (mean \u00b1 std across subjects) \u2014 {LABEL}",
    fontsize=13,
    y=1.01,
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "ica_temporal_profiles.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

---
## Analysis (c) — Frequency × Time Mean-Loading Heatmap

For each ICA component, we compute the **mean loading per subject**
at each wavelet frequency and time point.  The IC channel weights
are obtained from the scores `(F, C, K)` by averaging over channels,
and subject weights from the components `(K, S, T)` by averaging
over time:

```
freq_weights(k, f) = mean_c[ scores_2d(f, c, k) ]
sub_weights(k, s) = mean_t[ components_2d(k, s, t) ]
loading(k, f, t) = freq_weights(k,f) × mean_s[
    sub_weights(k,s) × mean_c[ bb_z(s,c,f,t) ]
]
```

### How this differs from subject-freq-features (Approach 8)

In Approach 8, scores are `(S,F,K)` and components are `(K,C,T)`;
here scores are `(F,C,K)` and components are `(K,S,T)`.  The heatmap
still shows frequency × time activation, but the roles of subjects
and channels are swapped between scores and components.


In [ ]:
n_show = min(6, N_COMPONENTS_ICA)
# Frequency weights from scores: mean over channels
freq_weights = scores_2d.mean(axis=1)  # (F, K)
# Subject weights from components: mean over time
sub_weights = components_2d.mean(axis=2)  # (K, S)
# Weighted data: mean over channels, then weight by subject weights
bb_z_chan_avg = bb_z.mean(axis=1)  # (S, F, T)
# weighted_sub(k, f, t) = mean_s[ sub_weights(k,s) * bb_z_chan_avg(s,f,t) ]
weighted_sub = np.einsum("ks,sft->kft", sub_weights, bb_z_chan_avg) / n_subjects
# Final: loading(k, f, t) = freq_weights(f, k) * weighted_sub(k, f, t)
ft_loading = np.einsum("fk,kft->kft", freq_weights, weighted_sub)

fig, axes = plt.subplots(n_show, 1, figsize=(14, 3 * n_show), sharex=True)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    data_i = ft_loading[i]  # (F, T)
    vmin_s, vmax_s = np.percentile(data_i, 1), np.percentile(data_i, 99)
    ax.pcolormesh(
        time,
        FREQS,
        data_i,
        cmap="inferno",
        vmin=vmin_s,
        vmax=vmax_s,
    )
    ax.set_ylabel("Freq (Hz)")
    ax.set_title(f"IC {i + 1} \u2014 Freq \u00d7 Time Mean Loading", fontsize=10)

axes[-1].set_xlabel("Time (s)")
fig.suptitle(
    f"Frequency \u00d7 Time Mean Loading per IC \u2014 {LABEL}",
    fontsize=13,
    y=1.01,
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "ica_time_frequency.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")


---
## Analysis (d) — Mean and Variance of Component Channel Loadings as Topomaps

Channel loadings for each component are obtained from the ICA
**score** matrix by averaging over frequencies → `(C, K)`.  These
represent the **spatial fingerprint** of each subject-temporal mode.

To assess inter-individual variability, we combine the per-subject
temporal patterns from the component with the channel scores to compute
per-subject channel loadings `(S, C, K)`, then compute:

- **Mean** across subjects → `(C, K)` — the average spatial distribution.
- **Variance** across subjects → `(C, K)` — electrodes where the mode
  strength varies most between individuals.

### How this differs from subject-freq-features (Approach 8)

In the subject-freq-features notebook, channel loadings come from ICA
**components** (time-averaged from the `(C, T)` component map).  Here,
the channel information lives in the ICA **scores** (reshaped to
`(F, C, K)`, averaged over frequencies), and per-subject variation
comes from weighting by each subject's temporal component pattern.

In [ ]:
# Channel loadings from scores: average over frequencies → (C, K)
score_channel_loadings = scores_2d.mean(axis=0)  # (C, K)

# Per-subject channel loadings:
# Weight score channel pattern by each subject's mean temporal activation
# components_2d: (K, S, T) → mean over T → (K, S) → transpose → (S, K)
subject_mean_activation = components_2d.mean(axis=2).T  # (S, K)
# Per-subject channel loading: (S, K) × (C, K) → (S, C, K)
ica_channel_loadings = np.einsum(
    "sk,ck->sck", subject_mean_activation, score_channel_loadings
)  # (S, C, K)

# Mean and variance across subjects
ica_ch_mean = ica_channel_loadings.mean(axis=0)  # (C, K)
ica_ch_var = ica_channel_loadings.var(axis=0)  # (C, K)

# Get MNE Info for topomap
info = analyzers[LABEL].info
info = mne.pick_info(info, mne.pick_types(info, eeg=True))
if n_channels < len(info.ch_names):
    info = mne.pick_info(info, list(range(n_channels)))

n_show = min(6, N_COMPONENTS_ICA)

# --- Mean topomaps ---
_vlim_mean = np.percentile(np.abs(ica_ch_mean[:, :n_show]), 99)
fig_mean, axes_mean = plt.subplots(1, n_show, figsize=(3.5 * n_show, 4))
if n_show == 1:
    axes_mean = [axes_mean]

for i, ax in enumerate(axes_mean):
    im, _ = plot_topomap(
        ica_ch_mean[:, i],
        info,
        axes=ax,
        show=False,
        cmap="RdBu_r",
        vlim=(-_vlim_mean, _vlim_mean),
    )
    ax.set_title(f"IC {i + 1}", fontsize=10)

fig_mean.suptitle(
    f"Mean Component Channel Loading (topomap) \u2014 {LABEL}",
    fontsize=12,
)
plt.colorbar(im, ax=axes_mean[-1], label="mean loading")
fig_mean.tight_layout()
if SAVE_PLOTS:
    fig_mean.savefig(PLOTS_DIR / "ica_topomap_mean.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

# --- Variance topomaps ---
_vmax_var = np.percentile(ica_ch_var[:, :n_show], 99)
fig_var, axes_var = plt.subplots(1, n_show, figsize=(3.5 * n_show, 4))
if n_show == 1:
    axes_var = [axes_var]

for i, ax in enumerate(axes_var):
    im, _ = plot_topomap(
        ica_ch_var[:, i],
        info,
        axes=ax,
        show=False,
        cmap="YlOrRd",
        vlim=(0, _vmax_var),
    )
    ax.set_title(f"IC {i + 1}", fontsize=10)

fig_var.suptitle(
    f"Variance of Component Channel Loading (topomap) \u2014 {LABEL}",
    fontsize=12,
)
plt.colorbar(im, ax=axes_var[-1], label="variance")
fig_var.tight_layout()
if SAVE_PLOTS:
    fig_var.savefig(
        PLOTS_DIR / "ica_topomap_variance.png", dpi=150, bbox_inches="tight"
    )
plt.show()
plt.close("all")

---
## Analysis (e) — Per-Subject Loading Bar Plot for Each Component

For each ICA component, compute the **mean absolute temporal activation**
per subject.  Since the component is shaped `(S, T)`, we take the mean
of the absolute values over time for each subject.  This scalar
summarises how strongly each participant expresses the subject-temporal
mode across the entire recording.

Subjects with uniformly high loadings indicate a stimulus-driven mode;
uneven loadings may reflect individual differences.

### How this differs from subject-freq-features (Approach 8)

In the subject-freq-features notebook, subject loadings are the mean
absolute ICA score over frequencies.  Here, the subject dimension lives
in the component pattern rather than the scores, so subject loadings
come from the time-averaged absolute component values.  This is a more
direct measure of each subject's temporal engagement with the mode.

In [ ]:
# Subject loadings: mean |component value| over time
subject_loadings = np.abs(components_2d).mean(axis=2).T  # (S, K)

n_show = min(6, N_COMPONENTS_ICA)
fig, axes = plt.subplots(1, n_show, figsize=(3 * n_show, 4), sharey=True)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    ax.barh(
        range(n_subjects),
        subject_loadings[:, i],
        color="darkorange",
    )
    ax.set_yticks(range(n_subjects))
    ax.set_yticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=8)
    ax.set_xlabel("|activation|")
    ax.set_title(f"IC {i + 1}", fontsize=10)

axes[0].set_ylabel("Subject")
fig.suptitle(
    f"Per-Subject Loading per Component \u2014 {LABEL}",
    fontsize=13,
    y=1.02,
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "ica_subject_loadings.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

---
## Summary

### Decomposition overview

This notebook uses the reshape `(F × C, S × T)` — frequencies and
channels as observations, subjects and time as features.  ICA
discovers **subject-temporal modes** (subject × time patterns) that
recur across frequencies and channels.

| Aspect | Subject–Freq Features (Approach 8) | Freq–Channel Features (here) |
|--------|--------------------------------------|-------------------------------|
| Observations | S × F | F × C |
| Features | C × T | S × T |
| Components represent | Spatial-temporal modes `(C, T)` | Subject-temporal modes `(S, T)` |
| Scores represent | Per-subject-freq weights `(S, F, K)` | Per-freq-channel weights `(F, C, K)` |
| Subject info lives in | Scores (axis 0) | Components (axis 1) |
| ISC measures | Spectral consistency | Temporal consistency |

The key advantage of this reshape is that subjects and time appear
together in the component pattern, allowing ICA to find **individual
response fingerprints** — how each subject's temporal activation
pattern generalises across frequency bands and scalp locations.

### Variables available for further analysis

| Variable | Shape | Description |
|----------|-------|-------------|
| `bb_z` | `(S, C, F, T)` | Z-scored 4-D wavelet power tensor |
| `X_fc` | `(F×C, S×T)` | Z-scored reshaped 2-D matrix |
| `pca` | — | Fitted PCA object |
| `pca_scores` | `(F×C, K_pca)` | PCA-transformed scores |
| `ica` | — | Fitted FastICA object |
| `ica_scores` | `(F×C, K_ica)` | ICA scores (per-observation weights) |
| `ica_components` | `(K_ica, S×T)` | ICA subject-temporal component patterns |
| `scores_2d` | `(F, C, K)` | ICA scores reshaped to frequency × channel |
| `components_2d` | `(K, S, T)` | ICA components reshaped to subject × time |
| `subject_loadings` | `(S, K)` | Per-subject mean |component| over time |
| `ica_channel_loadings` | `(S, C, K)` | Per-subject channel loadings |

### Analyses implemented

| # | Analysis | Key finding |
|---|----------|-------------|
| (a) | Intersubject correlation matrix | Which modes have consistent temporal profiles across subjects |
| (b) | Temporal profiles (mean ± std) | When each mode is active and how variable across subjects |
| (c) | Freq × Time mean loading | Mean IC loading at each wavelet frequency and time |
| (d) | Mean / variance topomaps | Spatial distribution and inter-individual variability |
| (e) | Per-subject loading bars | Individual-level contribution to each mode |

See `README.md` in this directory for the full analysis rationale,
alternative decomposition strategies, and ideas for future extensions.